# Embeddings From First Principles

Neural networks only work with numbers. Words are symbols. This notebook builds the bridge — from raw text to tokenization to learned embeddings — from scratch.

## The Obvious Approach — Just Use a Dictionary

The simplest idea: assign each word a unique number. But what happens when we encounter a word that's not in our dictionary?

In [1]:
# The whole-word approach: just assign each word a number
dictionary = ["cat", "dog", "car", "fish", "the", "sat", "on", "mat"]
word_to_id = {word: i for i, word in enumerate(dictionary)}

print(f"Our dictionary ({len(dictionary)} words):")
for word, idx in word_to_id.items():
    print(f"  {word:>4s} → {idx}")

# Now try to look up words not in the dictionary
test_words = ["cat", "unfairly", "ChatGPT", "running"]
print(f"\nLooking up words:")
for word in test_words:
    if word in word_to_id:
        print(f"  {word:>10s} → {word_to_id[word]}")
    else:
        print(f"  {word:>10s} → ??? (not in dictionary)")

print("\nAny word not in the dictionary is invisible to the model.")
print("And language keeps creating new words. The vocabulary will never be complete.")

Our dictionary (8 words):
   cat → 0
   dog → 1
   car → 2
  fish → 3
   the → 4
   sat → 5
    on → 6
   mat → 7

Looking up words:
         cat → 0
    unfairly → ??? (not in dictionary)
     ChatGPT → ??? (not in dictionary)
     running → ??? (not in dictionary)

Any word not in the dictionary is invisible to the model.
And language keeps creating new words. The vocabulary will never be complete.


## What's a Token?

Characters give us universal coverage but no meaning. Whole words give us meaning but break on anything unseen. Let's look at both extremes.

In [2]:
# Character-level tokenization
text = "the cat sat on the mat"
char_tokens = list(text.replace(" ", ""))
char_vocab = sorted(set(char_tokens))

print(f"Text: \"{text}\"")
print(f"Character tokens: {char_tokens}")
print(f"Vocabulary ({len(char_vocab)} tokens): {char_vocab}")
print(f"\nEvery possible word can be built from these. But does 'a' mean something?")
print("Does 't'? Individual characters carry no meaning.")

Text: "the cat sat on the mat"
Character tokens: ['t', 'h', 'e', 'c', 'a', 't', 's', 'a', 't', 'o', 'n', 't', 'h', 'e', 'm', 'a', 't']
Vocabulary (9 tokens): ['a', 'c', 'e', 'h', 'm', 'n', 'o', 's', 't']

Every possible word can be built from these. But does 'a' mean something?
Does 't'? Individual characters carry no meaning.


## BPE — Letting the Data Decide

BPE starts from characters and greedily merges the most frequent adjacent pair, over and over, until the vocabulary reaches a target size. Let's watch it happen step by step.

In [3]:
from collections import Counter

corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat chased the fish",
    "the dog chased the cat",
    "a car drove on the road",
    "a truck drove on the highway",
    "the fish swam in the pond",
    "the cat and dog played outside",
    "a car and truck were parked",
    "the dog ran across the yard",
    "the cat ran across the room",
]

def train_bpe(corpus, num_merges=15):
    """Train BPE tokenizer and show each merge step."""
    # Step 0: split every word into characters
    word_freqs = Counter()
    for sentence in corpus:
        for word in sentence.lower().split():
            word_freqs[tuple(word)] += 1
    
    merge_rules = []
    
    print(f"Initial vocabulary: {sorted(set(c for word in word_freqs for c in word))}\n")
    
    for step in range(num_merges):
        # Count all adjacent pairs
        pair_counts = Counter()
        for word, freq in word_freqs.items():
            for i in range(len(word) - 1):
                pair_counts[(word[i], word[i+1])] += freq
        
        if not pair_counts:
            break
        
        # Find the most frequent pair
        best_pair = pair_counts.most_common(1)[0]
        pair, count = best_pair
        merged = pair[0] + pair[1]
        merge_rules.append(pair)
        
        print(f"Merge {step+1:>2d}: ('{pair[0]}', '{pair[1]}') → '{merged}'  (count: {count})")
        
        # Apply the merge to all words
        new_word_freqs = Counter()
        for word, freq in word_freqs.items():
            new_word = []
            i = 0
            while i < len(word):
                if i < len(word) - 1 and word[i] == pair[0] and word[i+1] == pair[1]:
                    new_word.append(merged)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word_freqs[tuple(new_word)] += freq
        word_freqs = new_word_freqs
    
    # Show final vocabulary
    vocab = set()
    for word in word_freqs:
        for token in word:
            vocab.add(token)
    
    print(f"\nFinal vocabulary ({len(vocab)} tokens): {sorted(vocab, key=lambda x: (len(x), x))}")
    print(f"Merge rules: {merge_rules}")
    
    return word_freqs, merge_rules, vocab

word_freqs, merge_rules, vocab = train_bpe(corpus, num_merges=15)

Initial vocabulary: ['a', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y']

Merge  1: ('t', 'h') → 'th'  (count: 17)
Merge  2: ('th', 'e') → 'the'  (count: 17)
Merge  3: ('a', 't') → 'at'  (count: 8)
Merge  4: ('r', 'o') → 'ro'  (count: 6)
Merge  5: ('c', 'at') → 'cat'  (count: 5)
Merge  6: ('o', 'n') → 'on'  (count: 5)
Merge  7: ('d', 'o') → 'do'  (count: 4)
Merge  8: ('do', 'g') → 'dog'  (count: 4)
Merge  9: ('e', 'd') → 'ed'  (count: 4)
Merge 10: ('a', 'r') → 'ar'  (count: 4)
Merge 11: ('a', 'n') → 'an'  (count: 4)
Merge 12: ('r', 'u') → 'ru'  (count: 3)
Merge 13: ('s', 'at') → 'sat'  (count: 2)
Merge 14: ('c', 'h') → 'ch'  (count: 2)
Merge 15: ('ch', 'a') → 'cha'  (count: 2)

Final vocabulary (33 tokens): ['a', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'an', 'ar', 'at', 'ed', 'on', 'ro', 'ru', 'cat', 'cha', 'dog', 'sat', 'the']
Merge rules: [('t', 'h'), ('th', 'e'), ('a', 't

## The Ambiguity Problem

After training, our vocabulary contains overlapping tokens — "t", "h", "th", "the" might all exist. A word could theoretically be segmented multiple ways. The merge rules resolve this: applied in order, they produce exactly one tokenization per word.

In [4]:
def tokenize_bpe(word, merge_rules):
    """Tokenize a word using trained BPE merge rules (applied in order)."""
    tokens = list(word.lower())
    
    for pair in merge_rules:
        merged = pair[0] + pair[1]
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i+1] == pair[1]:
                new_tokens.append(merged)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    
    return tokens

# Tokenize some words — same merge rules, deterministic output
test_words = ["the", "cat", "chased", "across", "truck", "highway", "outside"]
print("Tokenization using trained BPE merge rules:\n")
for word in test_words:
    tokens = tokenize_bpe(word, merge_rules)
    print(f"  {word:>10s} → {tokens}")

print("\nSame word, same merge rules, same tokens. Every time.")
print("The merge rules are what make the tokenizer deterministic.")

Tokenization using trained BPE merge rules:

         the → ['the']
         cat → ['cat']
      chased → ['cha', 's', 'ed']
      across → ['a', 'c', 'ro', 's', 's']
       truck → ['t', 'ru', 'c', 'k']
     highway → ['h', 'i', 'g', 'h', 'w', 'a', 'y']
     outside → ['o', 'u', 't', 's', 'i', 'd', 'e']

Same word, same merge rules, same tokens. Every time.
The merge rules are what make the tokenizer deterministic.


## From Tokens to Vectors — The One-Hot Dead End

Now we have tokens. The obvious next step: assign each token a unique index and create a binary vector. Let's see why this fails.

In [5]:
import numpy as np
from itertools import combinations

# A vocabulary of 4 tokens
tokens = ['cat', 'dog', 'car', 'fish']
V = len(tokens)

# One-hot encode each token
one_hot = {token: np.eye(V)[i] for i, token in enumerate(tokens)}

print("One-hot encoding:\n")
for token, vec in one_hot.items():
    print(f"  {token:>4s} = {vec.astype(int).tolist()}")

# Compute the distance between every pair
print("\nEuclidean distances between all pairs:\n")
for t1, t2 in combinations(tokens, 2):
    dist = np.linalg.norm(one_hot[t1] - one_hot[t2])
    print(f"  d({t1:>4s}, {t2:<4s}) = {dist:.4f}")

print(f"\nEvery pair is equidistant at sqrt(2) = {np.sqrt(2):.4f}.")
print("The representation carries no information about token relationships.")

ModuleNotFoundError: No module named 'numpy'

## Section 4: The Distributional Hypothesis — Context as Meaning

If two words keep appearing in the same neighborhoods, they probably share meaning. Let's measure context overlap between words in our corpus.

In [ ]:
def get_context_windows(corpus, window_size=2):
    """Extract (center_word, context_word) pairs from the corpus."""
    pairs = []
    for sentence in corpus:
        words = sentence.lower().split()
        for i, center in enumerate(words):
            for j in range(max(0, i - window_size), min(len(words), i + window_size + 1)):
                if i != j:
                    pairs.append((center, words[j]))
    return pairs

pairs = get_context_windows(corpus, window_size=2)

# Show context windows for a few words
for target in ['cat', 'dog', 'car']:
    contexts = [ctx for center, ctx in pairs if center == target]
    print(f"'{target}' appears near: {contexts}")
    print()

# Measure context overlap
def context_overlap(word1, word2, pairs):
    """What fraction of context words do two words share?"""
    ctx1 = Counter(ctx for center, ctx in pairs if center == word1)
    ctx2 = Counter(ctx for center, ctx in pairs if center == word2)
    shared = set(ctx1.keys()) & set(ctx2.keys())
    all_ctx = set(ctx1.keys()) | set(ctx2.keys())
    return len(shared) / len(all_ctx) if all_ctx else 0

test_pairs = [('cat', 'dog'), ('cat', 'car'), ('car', 'truck'), ('dog', 'truck')]

print("Context overlap (Jaccard similarity):\n")
for w1, w2 in test_pairs:
    overlap = context_overlap(w1, w2, pairs)
    print(f"  {w1:>5s} <-> {w2:<5s} : {overlap:.2f}")

print("\nWords with high context overlap should end up close in embedding space.")
print("Words with low overlap should end up far apart.")

## Section 5: Word2Vec — Skip-Gram Training Pairs

Word2Vec's skip-gram model pairs each center word with its context words (positive pairs) and random non-context words (negative pairs). The center's input embedding gets pulled toward context and pushed away from negatives.

In [ ]:
import random

# Build vocabulary from corpus
all_words = set()
for sentence in corpus:
    all_words.update(sentence.lower().split())
vocab = sorted(all_words)
word2idx = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

print(f"Vocabulary ({V} words): {vocab}\n")

# Show skip-gram training pairs with gradient directions
print("Skip-gram training example:\n")
sentence = corpus[0]
words = sentence.lower().split()
print(f"  Sentence: \"{sentence}\"")
print(f"  Center word: '{words[1]}' (window=2)\n")

center = words[1]
context_words = []
for j in range(max(0, 1-2), min(len(words), 1+2+1)):
    if j != 1:
        context_words.append(words[j])
        print(f"  Positive: ({center}, {words[j]})  →  pull {center}'s input toward {words[j]}'s output")

random.seed(42)
neg_samples = random.sample([w for w in vocab if w not in [center] + context_words], 5)
print()
for neg in neg_samples:
    print(f"  Negative: ({center}, {neg:>8s})  →  push {center}'s input away from {neg}'s output")

print(f"\n  {center}'s input embedding: updated by ALL gradients (pull + push)")
print(f"  Context outputs ({', '.join(context_words)}): updated by pull gradient only")
print(f"  Negative outputs ({', '.join(neg_samples)}): updated by push gradient only")

## The Ablations

Three questions, three experiments:

1. **Does the sigmoid matter?** Train with sigmoid loss vs linear loss. If both learn the same structure, the sigmoid doesn't contaminate the geometry.
2. **Input vs output embeddings:** Do both vectors learn the same thing? If a word has one meaning, they should agree.
3. **One vector vs two:** Train with a single embedding per word (used for both center and context roles). Does it learn the same structure as the two-vector approach?

In [ ]:
def sigmoid(x):
    """Numerically stable sigmoid."""
    return np.where(x >= 0,
                    1 / (1 + np.exp(-x)),
                    np.exp(x) / (1 + np.exp(x)))

class Word2Vec:
    def __init__(self, vocab_size, embed_dim, use_sigmoid=True, single_embedding=False):
        self.W_in = np.random.randn(vocab_size, embed_dim) * 0.1
        self.single_embedding = single_embedding
        if single_embedding:
            self.W_out = self.W_in  # same matrix — one vector per word
        else:
            self.W_out = np.random.randn(vocab_size, embed_dim) * 0.1
        self.use_sigmoid = use_sigmoid
    
    def train_step(self, center_idx, context_idx, neg_indices, lr):
        v_center = self.W_in[center_idx].copy()
        u_context = self.W_out[context_idx].copy()
        
        if self.use_sigmoid:
            pos_score = np.dot(v_center, u_context)
            pos_sig = sigmoid(pos_score)
            pos_loss = -np.log(pos_sig + 1e-10)
            pos_grad_center = (pos_sig - 1) * u_context
            pos_grad_context = (pos_sig - 1) * v_center
            
            neg_loss = 0
            neg_grad_center = np.zeros_like(v_center)
            for neg_idx in neg_indices:
                u_neg = self.W_out[neg_idx].copy()
                neg_score = np.dot(v_center, u_neg)
                neg_sig = sigmoid(neg_score)
                neg_loss += -np.log(1 - neg_sig + 1e-10)
                neg_grad_center += neg_sig * u_neg
                self.W_out[neg_idx] -= lr * (neg_sig * v_center)
            
            loss = pos_loss + neg_loss
        else:
            pos_score = np.dot(v_center, u_context)
            pos_loss = (1 - pos_score) ** 2
            pos_grad_center = -2 * (1 - pos_score) * u_context
            pos_grad_context = -2 * (1 - pos_score) * v_center
            
            neg_loss = 0
            neg_grad_center = np.zeros_like(v_center)
            for neg_idx in neg_indices:
                u_neg = self.W_out[neg_idx].copy()
                neg_score = np.dot(v_center, u_neg)
                neg_loss += neg_score ** 2
                neg_grad_center += 2 * neg_score * u_neg
                self.W_out[neg_idx] -= lr * (2 * neg_score * v_center)
            
            loss = pos_loss + neg_loss
        
        self.W_in[center_idx] -= lr * (pos_grad_center + neg_grad_center)
        if not self.single_embedding:
            self.W_out[context_idx] -= lr * pos_grad_context
        
        return loss

print("Word2Vec model defined with three modes:")
print("  1. Two embeddings + sigmoid loss (standard)")
print("  2. Two embeddings + linear loss (sigmoid ablation)")
print("  3. Single embedding + sigmoid loss (embedding ablation)")

In [ ]:
def train_word2vec(corpus, embed_dim=20, window_size=2, n_neg=5, lr=0.025, epochs=100,
                   use_sigmoid=True, single_embedding=False, seed=42):
    """Train Word2Vec skip-gram with negative sampling."""
    np.random.seed(seed)
    random.seed(seed)
    
    all_words = set()
    for sentence in corpus:
        all_words.update(sentence.lower().split())
    vocab = sorted(all_words)
    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}
    V = len(vocab)
    
    training_pairs = []
    for sentence in corpus:
        words = sentence.lower().split()
        for i, center in enumerate(words):
            for j in range(max(0, i - window_size), min(len(words), i + window_size + 1)):
                if i != j:
                    training_pairs.append((word2idx[center], word2idx[words[j]]))
    
    model = Word2Vec(V, embed_dim, use_sigmoid=use_sigmoid, single_embedding=single_embedding)
    
    losses = []
    for epoch in range(epochs):
        random.shuffle(training_pairs)
        epoch_loss = 0
        for center_idx, context_idx in training_pairs:
            neg_indices = []
            while len(neg_indices) < n_neg:
                neg = random.randint(0, V - 1)
                if neg != center_idx and neg != context_idx:
                    neg_indices.append(neg)
            loss = model.train_step(center_idx, context_idx, neg_indices, lr)
            epoch_loss += loss
        
        avg_loss = epoch_loss / len(training_pairs)
        losses.append(avg_loss)
        if (epoch + 1) % 25 == 0:
            print(f"  Epoch {epoch+1:>3d}/{epochs}: avg loss = {avg_loss:.4f}")
    
    return model, vocab, word2idx, idx2word, losses

# Train all three variants
print("=" * 55)
print("  MODEL 1: Two embeddings + sigmoid (standard)")
print("=" * 55)
model_std, vocab, word2idx, idx2word, losses_std = train_word2vec(
    corpus, use_sigmoid=True, single_embedding=False
)

print(f"\n{'=' * 55}")
print("  MODEL 2: Two embeddings + linear (sigmoid ablation)")
print("=" * 55)
model_lin, _, _, _, losses_lin = train_word2vec(
    corpus, use_sigmoid=False, single_embedding=False
)

print(f"\n{'=' * 55}")
print("  MODEL 3: Single embedding + sigmoid (embedding ablation)")
print("=" * 55)
model_single, _, _, _, losses_single = train_word2vec(
    corpus, use_sigmoid=True, single_embedding=True
)

In [ ]:
import matplotlib.pyplot as plt

# Loss curves for all three
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, losses, title, color in zip(axes,
    [losses_std, losses_lin, losses_single],
    ['Two emb + sigmoid\n(standard)', 'Two emb + linear\n(sigmoid ablation)', 'Single emb + sigmoid\n(embedding ablation)'],
    ['tab:blue', 'tab:orange', 'tab:green']):
    ax.plot(losses, linewidth=2, color=color)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Average Loss')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Ablation 1: Does the sigmoid matter?

Comparing cosine similarities and nearest neighbors between sigmoid and linear loss models. If both learn the same structure, the sigmoid doesn't contaminate the geometry.

In [ ]:
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)

def get_nearest_neighbor(word, embeddings, word2idx, vocab, exclude_stopwords=True):
    """Find the nearest neighbor of a word by cosine similarity."""
    stop = ['the', 'a', 'on', 'in', 'and', 'were'] if exclude_stopwords else []
    best_word, best_sim = "", -2
    for w2 in vocab:
        if w2 == word or w2 in stop or w2 not in word2idx:
            continue
        sim = cosine_sim(embeddings[word2idx[word]], embeddings[word2idx[w2]])
        if sim > best_sim:
            best_word, best_sim = w2, sim
    return best_word, best_sim

def compare_two_models(emb_a, emb_b, name_a, name_b, word2idx, vocab):
    """Compare cosine similarities between two embedding sets."""
    similar = [('cat','dog'), ('car','truck'), ('cat','fish'), ('ran','chased')]
    dissimilar = [('cat','car'), ('dog','truck'), ('cat','road'), ('fish','drove')]
    content = [w for w in vocab if w not in ['the','a','on','in','and','were']]
    
    print(f"\n  {'Pair':>18s}  {name_a:>10s}  {name_b:>10s}")
    print(f"  {'─'*18}  {'─'*10}  {'─'*10}")
    
    print("  Expected SIMILAR:")
    for w1, w2 in similar:
        if w1 in word2idx and w2 in word2idx:
            sa = cosine_sim(emb_a[word2idx[w1]], emb_a[word2idx[w2]])
            sb = cosine_sim(emb_b[word2idx[w1]], emb_b[word2idx[w2]])
            print(f"  {w1:>7s} <-> {w2:<7s}  {sa:>+10.3f}  {sb:>+10.3f}")
    
    print("  Expected DISSIMILAR:")
    for w1, w2 in dissimilar:
        if w1 in word2idx and w2 in word2idx:
            sa = cosine_sim(emb_a[word2idx[w1]], emb_a[word2idx[w2]])
            sb = cosine_sim(emb_b[word2idx[w1]], emb_b[word2idx[w2]])
            print(f"  {w1:>7s} <-> {w2:<7s}  {sa:>+10.3f}  {sb:>+10.3f}")
    
    print(f"\n  {'Word':>10s}  {name_a+' NN':>12s}  {name_b+' NN':>12s}  {'Same?':>6s}")
    print(f"  {'─'*10}  {'─'*12}  {'─'*12}  {'─'*6}")
    for w in content:
        nn_a = get_nearest_neighbor(w, emb_a, word2idx, vocab)
        nn_b = get_nearest_neighbor(w, emb_b, word2idx, vocab)
        same = "YES" if nn_a[0] == nn_b[0] else "no"
        print(f"  {w:>10s}  {nn_a[0]:>12s}  {nn_b[0]:>12s}  {same:>6s}")

print("=" * 55)
print("  ABLATION 1: Sigmoid vs Linear Loss")
print("  (both use two embeddings, comparing input embeddings)")
print("=" * 55)
compare_two_models(model_std.W_in, model_lin.W_in, "Sigmoid", "Linear", word2idx, vocab)

## Ablation 2: Input vs Output Embeddings

A word has one meaning. Do both embedding matrices learn the same thing?

In [ ]:
print("=" * 55)
print("  ABLATION 2: Input vs Output Embeddings")
print("  (same model, comparing the two embedding matrices)")
print("=" * 55)
compare_two_models(model_std.W_in, model_std.W_out, "Input", "Output", word2idx, vocab)

## Ablation 3: Two Vectors vs One Vector

Does using a single embedding per word (same vector for center and context roles) learn the same structure as the standard two-vector approach?

In [ ]:
print("=" * 55)
print("  ABLATION 3: Two Vectors vs One Vector")
print("  (standard input emb vs single-embedding model)")
print("=" * 55)
compare_two_models(model_std.W_in, model_single.W_in, "Two-vec", "Single", word2idx, vocab)

## 2D Visualization — All Models Side by Side

PCA projection to 2D. Clusters should be visible; fine-grained structure might not survive the projection.

In [ ]:
from sklearn.decomposition import PCA

def plot_embeddings(embeddings, vocab, word2idx, title, ax):
    content_words = [w for w in vocab if w not in ['the', 'a', 'on', 'in', 'and', 'were']]
    indices = [word2idx[w] for w in content_words]
    vecs = embeddings[indices]
    
    pca = PCA(n_components=2)
    coords = pca.fit_transform(vecs)
    
    ax.scatter(coords[:, 0], coords[:, 1], s=80, alpha=0.7, edgecolors='black', linewidth=0.5)
    for i, word in enumerate(content_words):
        ax.annotate(word, (coords[i, 0], coords[i, 1]),
                   fontsize=9, fontweight='bold',
                   xytext=(5, 5), textcoords='offset points')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.0%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.0%})')
    ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

plot_embeddings(model_std.W_in, vocab, word2idx, 'Standard (input emb)', axes[0, 0])
plot_embeddings(model_std.W_out, vocab, word2idx, 'Standard (output emb)', axes[0, 1])
plot_embeddings(model_lin.W_in, vocab, word2idx, 'Linear loss (input emb)', axes[1, 0])
plot_embeddings(model_single.W_in, vocab, word2idx, 'Single embedding', axes[1, 1])

plt.suptitle('Do all approaches learn the same structure?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Top-left:  Standard model, input embeddings (what we normally keep)")
print("Top-right: Standard model, output embeddings (what we normally throw away)")
print("Bot-left:  Linear loss model (sigmoid ablation)")
print("Bot-right: Single embedding model (one vector per word)")